# Interplanetary Survival Guide: Martian Map — Complete 6-Step Pipeline & 3x3 Chunk Migration Engine
**NASA Space Apps Challenge | Technical Architecture Notebook**

This notebook implements the complete mathematical and algorithmic pipeline for autonomous astronaut & rover EVA route planning on Lower Mount Sharp, Gale Crater, Mars. It loads the preloaded $3 \times 3$ chunk dataset generated by `nasa_mars_chunk_loader.py`, executes **Steps 1 through 6** starting on the Center Chunk $(1, 1)$, dynamically migrates and stitches the adjacent chunk when the astronaut selects a destination outside the center chunk, and exports a unified **CesiumJS 3D Mission Bundle (`cesium_mars_mission_bundle.json`)**.

---
### Step 0 & Step 1: Terrain Geometry ($\theta, R$), Material Friction $\eta(r,c)$, & Cost Field $C(r,c)$
We load the $3 \times 3$ preloaded chunk mosaic from `mars_3x3_chunks_ready.npz`. On the active chunk $(I, J)$, the computationally optimized equations are:
1. **Squared Central-Difference Slope & Cliff Mask ($>20^\circ$):**
   $$G^2(r,c) = \left(\frac{z(r+1,c)-z(r-1,c)}{2\Delta y}\right)^2 + \left(\frac{z(r,c+1)-z(r,c-1)}{2\Delta x}\right)^2, \quad \mathcal{M}_{\text{pass}}(r,c) = \big(G^2(r,c) \le \tan^2 20^\circ\big)$$
2. **Denoised $5\times 5\text{ m}$ Separable Roughness:**
   $$R(r,c) = \sqrt{\max\!\left(0,\; \mu_{5\times 5}(z^2) - [\mu_{5\times 5}(z)]^2 - \sigma_z(r,c)^2\right)}$$
3. **Dimensionless Material Friction Multiplier:**
   $$\eta(r,c) = \eta_{\text{geo}}(r,c)\left[1 + \max\!\big(0,\; 4.8 - 0.03\,I_{\text{th}}(r,c)\big) + \frac{0.4}{R_{\text{ref}}}R(r,c)\right]$$
4. **Continuous Pixel Cost Field:**
   $$C(r,c) = \eta(r,c)\big(100 + 14\,\theta(r,c)\big) + 40\,U(r,c) + 60\big(1 - L_{\text{LOS}}(r,c)\big) \quad (\text{with } C = \infty \text{ on } \neg\mathcal{M}_{\text{pass}})$$

In [1]:
import heapq
import json
import math
import numpy as np
from scipy.ndimage import uniform_filter, distance_transform_edt, sobel, laplace
from scipy.spatial import KDTree

from nasa_mars_chunk_loader import (
    ChunkMigrationManager,
    pixel_to_lonlat,
    lonlat_to_pixel,
)

# Load the 3x3 preloaded chunk bundle prepared by nasa_mars_chunk_loader.py
bundle = np.load("mars_3x3_chunks_ready.npz", allow_pickle=True)
meta = json.loads(str(bundle["metadata_json"]))
rover_catalog = json.loads(str(bundle["rover_catalog_json"]))
arrays = {k: bundle[k] for k in bundle.files if k not in ("metadata_json", "rover_catalog_json")}

L = int(meta["chunk_size_L"])
h_global = int(meta["halo_h"])
G = int(meta["global_core_size"])
R_ref = float(meta["R_ref"])
S_grad = float(meta["S_grad"])
S_lap = float(meta["S_lap"])

mgr = ChunkMigrationManager(arrays, meta)
chunk_11 = mgr.slice_chunk_with_halo(1, 1, stencil_h=0)

print(f"Loaded Mission Site : {meta['mission_name']} ({G}m x {G}m 3x3 Mosaic, L={L}m/chunk)")
print(f"Global Constants    : R_ref = {R_ref:.4f} m | S_grad = {S_grad:.1f} | S_lap = {S_lap:.1f}")
print(f"Center Chunk (1,1)  : Passable = {100.0*np.mean(chunk_11['pass_pad']):.2f}% | "
      f"Finite Cost C range = [{chunk_11['C_pad'][chunk_11['pass_pad']].min():.1f}, "
      f"{chunk_11['C_pad'][chunk_11['pass_pad']].max():.1f}]")

Loaded Mission Site : MSL Curiosity (384m x 384m 3x3 Mosaic, L=128m/chunk)
Global Constants    : R_ref = 0.4712 m | S_grad = 3214.1 | S_lap = 2151.8
Center Chunk (1,1)  : Passable = 77.89% | Finite Cost C range = [130.6, 2964.9]

---
### Step 2: Shift-Invariant Sobel + Laplacian Deviation Score $D^{(I,J)}(r,c)$
Using the globally sanitized EDT scratch buffer $C_{\text{safe}}$ (Equation 2.1) and a minimal $h = 1\text{ px}$ stencil halo, we compute directional cost transitions via the separable $3\times 3$ **Sobel operator** (Equation 2.2) and localized cost changes via the $5\text{-point}$ **Discrete Laplacian** (Equation 2.3), normalized by fixed map-wide scales $(S_{\nabla}, S_{\nabla^2})$:

$$D^{(I,J)}(r,c) = \begin{cases} 0.0, & \text{if } \neg\mathcal{M}_{\text{pass}}^{(I,J)}(r,c) \\[6pt] 0.45\,\operatorname{clip}\!\left(\dfrac{|\nabla C|}{S_{\nabla}},0,1\right) + 0.30\,\operatorname{clip}\!\left(\dfrac{|\nabla^2 C|}{S_{\nabla^2}},0,1\right) + 0.25\,U^{(I,J)}(r,c), & \text{if } \mathcal{M}_{\text{pass}}^{(I,J)}(r,c) \end{cases}$$

In [2]:
def compute_chunk_deviation_score(mgr: ChunkMigrationManager, I: int, J: int) -> dict:
    """Computes Step 2 Deviation Score D^(I,J)(r,c) using only an h=1px halo slice."""
    win = mgr.slice_chunk_with_halo(I, J, stencil_h=1)
    C_safe_pad = win["C_safe_pad"]
    pass_pad = win["pass_pad"]
    pass_core = pass_pad[1:-1, 1:-1]
    U_core = win["U_pad"][1:-1, 1:-1]

    # Eq 2.2 & 2.3: Separable 3x3 Sobel & 5-tap Discrete Laplacian on C_safe_pad
    grad_core = np.hypot(sobel(C_safe_pad, axis=0), sobel(C_safe_pad, axis=1))[1:-1, 1:-1]
    lap_core = np.abs(laplace(C_safe_pad))[1:-1, 1:-1]

    grad_norm = np.clip(grad_core / S_grad, 0.0, 1.0)
    lap_norm = np.clip(lap_core / S_lap, 0.0, 1.0)

    # Eq 2.4: Globally normalized convex combination
    D_core = np.where(pass_core, 0.45 * grad_norm + 0.30 * lap_norm + 0.25 * U_core, 0.0)
    return {
        "chunk_ij": (I, J),
        "D": D_core,
        "pass_mask": pass_core,
        "z": win["z_pad"][1:-1, 1:-1],
        "eta": win["eta_pad"][1:-1, 1:-1],
        "C": win["C_pad"][1:-1, 1:-1],
        "U": U_core,
        "L_LOS": win["L_LOS_pad"][1:-1, 1:-1],
        "R_rad": win["R_rad_pad"][1:-1, 1:-1],
    }

chunk_data_11 = compute_chunk_deviation_score(mgr, 1, 1)
chunk_data_12 = compute_chunk_deviation_score(mgr, 1, 2)

print(f"Chunk (1,1) Deviation Score D range : [{chunk_data_11['D'].min():.3f}, {chunk_data_11['D'].max():.3f}]")
print(f"Chunk (1,2) Deviation Score D range : [{chunk_data_12['D'].min():.3f}, {chunk_data_12['D'].max():.3f}]")
print(f"Cliff pixels strictly zeroed in D   : {np.all(chunk_data_11['D'][~chunk_data_11['pass_mask']] == 0.0)}")

Chunk (1,1) Deviation Score D range : [0.000, 0.769]
Chunk (1,2) Deviation Score D range : [0.000, 0.328]
Cliff pixels strictly zeroed in D   : True

---
### Step 3: Passable-Only 1D CDF Vertex Sampling, `KDTree` Graph & $26\text{ m}$ Boundary-Strip Migration
1. **Passable-Only 1D CDF Binary Search (Equation 3.1):**
   $$F_k = \sum_{m=1}^{k} \big(D(r_m, c_m) + \epsilon_{\text{floor}}\big), \quad \epsilon_{\text{floor}} = 0.05$$
   We seed each chunk deterministically via spatial hash $\operatorname{seed}(I,J) = (I \cdot 73856093 \oplus J \cdot 19349663) \bmod 2^{32}$.
2. **Batched 11-Point Ray Validation & $26\text{ m}$ Boundary-Strip Stitching (Equation 3.2):**
   Candidate edges within $R = 26\text{ m}$ are validated in a single vectorized $(M \times 11 \times 2)$ tensor pass against $\mathcal{M}_{\text{pass}}$. When the astronaut selects a target in East Chunk $(1, 2)$, we stitch **only** the $26\text{ m}$ boundary strip between Chunk $(1, 1)$ and Chunk $(1, 2)$.

In [3]:
def sample_chunk_vertices_and_edges(
    chunk_info: dict,
    rover_wps: list,
    n_samples: int = 340,
    radius_m: float = 26.0,
    eps_floor: float = 0.05,
) -> dict:
    """Samples adaptive navigation vertices via 1D passable CDF and builds validated KDTree edges."""
    I, J = chunk_info["chunk_ij"]
    D = chunk_info["D"]
    pass_mask = chunk_info["pass_mask"]

    # Deterministic spatial chunk hash seed(I, J)
    seed_ij = ((I * 73856093) ^ (J * 19349663) ^ 42) & 0xFFFFFFFF
    rng = np.random.default_rng(seed_ij)

    # Eq 3.1: Passable-only 1D CDF binary search
    pass_rc = np.argwhere(pass_mask)
    weights = D[pass_mask] + eps_floor
    cdf = np.cumsum(weights)
    cdf /= cdf[-1]

    u_draws = (np.arange(n_samples, dtype=np.float64) + rng.random(n_samples)) / n_samples
    chosen_idx = np.unique(np.searchsorted(cdf, u_draws))
    local_rc = pass_rc[chosen_idx]

    # Pin any Rover / Science Waypoints located inside this chunk
    pinned_meta = {}
    for wp in rover_wps:
        if tuple(wp["chunk_ij"]) == (I, J):
            wp_rc = np.array(wp["local_rc"], dtype=int)
            if not pass_mask[wp_rc[0], wp_rc[1]]:
                dists = np.sum((pass_rc - wp_rc) ** 2, axis=1)
                wp_rc = pass_rc[np.argmin(dists)]
            local_rc = np.vstack([wp_rc, local_rc])
            pinned_meta[wp["id"]] = (int(wp_rc[0] + I * L), int(wp_rc[1] + J * L))

    _, uniq_idx = np.unique(local_rc, axis=0, return_index=True)
    local_rc = local_rc[np.sort(uniq_idx)]
    global_rc = local_rc + np.array([I * L, J * L])

    # Build KDTree and query pairs within R = 26m
    tree = KDTree(global_rc)
    pairs = tree.query_pairs(r=radius_m, output_type="ndarray")

    # Eq 3.2: Batched 11-point segment validation against pass_mask
    u_pts = local_rc[pairs[:, 0]]
    v_pts = local_rc[pairs[:, 1]]
    lambdas = np.linspace(0.0, 1.0, 11)[None, :, None]
    seg_rc = np.rint((1.0 - lambdas) * u_pts[:, None, :] + lambdas * v_pts[:, None, :]).astype(int)
    seg_rc = np.clip(seg_rc, 0, L - 1)

    valid_edge_mask = np.all(pass_mask[seg_rc[:, :, 0], seg_rc[:, :, 1]], axis=1)
    valid_pairs = pairs[valid_edge_mask]
    valid_seg_rc = seg_rc[valid_edge_mask]
    eta_bar = np.mean(chunk_info["eta"][valid_seg_rc[:, :, 0], valid_seg_rc[:, :, 1]], axis=1)

    return {
        "chunk_ij": (I, J),
        "local_rc": local_rc,
        "global_rc": global_rc,
        "tree": tree,
        "valid_pairs": valid_pairs,
        "eta_bar": eta_bar,
        "pinned_waypoints": pinned_meta,
    }

graph_11 = sample_chunk_vertices_and_edges(chunk_data_11, rover_catalog, n_samples=340)
graph_12 = sample_chunk_vertices_and_edges(chunk_data_12, rover_catalog, n_samples=340)

# Dynamic 26m Boundary-Strip Stitching between Center Chunk (1,1) and East Chunk (1,2)
seam_col_g = 2 * L
strip_11_idx = np.where(graph_11["global_rc"][:, 1] >= seam_col_g - 26)[0]
strip_12_idx = np.where(graph_12["global_rc"][:, 1] <= seam_col_g + 26)[0]

tree_strip_11 = KDTree(graph_11["global_rc"][strip_11_idx])
tree_strip_12 = KDTree(graph_12["global_rc"][strip_12_idx])
cross_neighbors = tree_strip_11.query_ball_tree(tree_strip_12, r=26.0)

pass_core_global = arrays["pass_pad"][h_global:-h_global, h_global:-h_global]
eta_core_global = arrays["eta_pad"][h_global:-h_global, h_global:-h_global]
bridge_edges = []
for i_local, nbr_list in enumerate(cross_neighbors):
    u_g = graph_11["global_rc"][strip_11_idx[i_local]]
    for j_local in nbr_list:
        v_g = graph_12["global_rc"][strip_12_idx[j_local]]
        pts = np.rint(np.linspace(u_g, v_g, 11)).astype(int)
        if np.all(pass_core_global[pts[:, 0], pts[:, 1]]):
            eta_uv = float(np.mean(eta_core_global[pts[:, 0], pts[:, 1]]))
            bridge_edges.append((tuple(int(x) for x in u_g), tuple(int(x) for x in v_g), eta_uv))

print(f"Center Chunk (1,1) : {len(graph_11['global_rc'])} vertices | {len(graph_11['valid_pairs'])*2} directed edges")
print(f"East Chunk   (1,2) : {len(graph_12['global_rc'])} vertices | {len(graph_12['valid_pairs'])*2} directed edges")
print(f"26m Seam Strip     : Queried only {len(strip_11_idx)}x{len(strip_12_idx)} border nodes -> {len(bridge_edges)*2} bridge edges stitched!")

Center Chunk (1,1) : 344 vertices | 12762 directed edges
East Chunk   (1,2) : 341 vertices | 11820 directed edges
26m Seam Strip     : Queried only 70x84 border nodes -> 1180 bridge edges stitched!

---
### Step 4: Mars Bioenergetic Pandolf Edge Weights & Once-Per-Leg Coefficient Precomputation
Rather than recomputing suit/payload constants on every edge, we precompute three scalars once per mission leg for astronaut mass $W = 80\text{ kg}$, payload $L = 65 + m_{\text{samples}}$, velocity $V = 1.1\text{ m/s}$, and Mars gravity $G_{\text{mars}} = 0.38$:
$$P_{\text{stand}}(L) = 0.38\left[1.5W + 2.0(W+L)\left(\frac{L}{W}\right)^2\right], \quad \alpha_V(L) = 0.57(W+L)V^2, \quad \beta_V(L) = 0.133(W+L)V$$
Each directed edge $(u \to v)$ evaluates its oxygen cost $\Delta\text{O}_2(u \to v)$ in liters directly, with the downhill braking floor $0.60\,\alpha_V(L)$:
$$P_{\text{watts}}(u \to v) = P_{\text{stand}}(L) + \bar{\eta}_{uv}\,\max\!\Big(0.60\,\alpha_V(L),\; \alpha_V(L) + \beta_V(L)\,S_{uv}\Big), \qquad \Delta\text{O}_2(u \to v) = \frac{P_{\text{watts}}(u \to v)\,d_{3D}(u,v)}{20,100\,V}$$

In [4]:
z_core_global = arrays["z_pad"][h_global:-h_global, h_global:-h_global]

class DirectedMarsGraph:
    """High-speed adjacency-list Directed Graph (DiGraph) supporting forward A* & reversed Dijkstra."""
    def __init__(self):
        self.nodes = {}      # u -> {"z": float, "chunk_ij": (I, J)}
        self.adj = {}        # u -> {v: edge_dict}
        self.rev_adj = {}    # v -> {u: edge_dict}  (Implicit G.reverse() view!)

    def add_node(self, u: tuple, z: float, chunk_ij: tuple):
        if u not in self.nodes:
            self.nodes[u] = {"z": z, "chunk_ij": chunk_ij}
            self.adj[u] = {}
            self.rev_adj[u] = {}

    def add_symmetric_pair(self, u: tuple, v: tuple, eta_val: float):
        d_2d = math.hypot(u[0] - v[0], u[1] - v[1])
        dz = self.nodes[v]["z"] - self.nodes[u]["z"]
        d_3d = math.hypot(d_2d, dz)
        S_uv = 100.0 * (dz / max(d_2d, 1e-6))
        e_fwd = {"d_2d": d_2d, "d_3d": d_3d, "S_grade": S_uv,  "eta_bar": eta_val, "O2_liters": 0.0}
        e_rev = {"d_2d": d_2d, "d_3d": d_3d, "S_grade": -S_uv, "eta_bar": eta_val, "O2_liters": 0.0}
        self.adj[u][v] = e_fwd
        self.rev_adj[v][u] = e_fwd
        self.adj[v][u] = e_rev
        self.rev_adj[u][v] = e_rev

    def number_of_nodes(self) -> int:
        return len(self.nodes)

    def number_of_edges(self) -> int:
        return sum(len(nbrs) for nbrs in self.adj.values())

def build_stitched_digraph(graphs: list, bridges: list) -> DirectedMarsGraph:
    G_dir = DirectedMarsGraph()
    for g in graphs:
        for r_g, c_g in g["global_rc"]:
            u = (int(r_g), int(c_g))
            G_dir.add_node(u, z=float(z_core_global[r_g, c_g]), chunk_ij=g["chunk_ij"])
        for (idx_u, idx_v), eta_val in zip(g["valid_pairs"], g["eta_bar"]):
            u = tuple(int(x) for x in g["global_rc"][idx_u])
            v = tuple(int(x) for x in g["global_rc"][idx_v])
            G_dir.add_symmetric_pair(u, v, float(eta_val))
    for u, v, eta_val in bridges:
        G_dir.add_symmetric_pair(u, v, float(eta_val))
    return G_dir

def update_graph_o2_weights_for_payload(
    G_dir: DirectedMarsGraph, m_samples_kg: float = 0.0, W_kg: float = 80.0, V_ms: float = 1.1
) -> dict:
    """Step 4 Opt 1 & 2: Precomputes payload scalars once per leg and updates all edge O2 weights."""
    L_kg = 65.0 + m_samples_kg
    G_mars = 0.38
    P_stand = G_mars * (1.5 * W_kg + 2.0 * (W_kg + L_kg) * ((L_kg / W_kg) ** 2))
    alpha_V = G_mars * 1.5 * (W_kg + L_kg) * (V_ms ** 2)
    beta_V = G_mars * 0.35 * (W_kg + L_kg) * V_ms
    gamma_O2 = 1.0 / (20_100.0 * V_ms)

    for u, nbrs in G_dir.adj.items():
        for v, data in nbrs.items():
            work_term = max(0.60 * alpha_V, alpha_V + beta_V * data["S_grade"])
            P_watts = P_stand + data["eta_bar"] * work_term
            data["P_watts"] = P_watts
            data["O2_liters"] = gamma_O2 * P_watts * data["d_3d"]

    return {
        "L_kg": L_kg,
        "P_stand": P_stand,
        "alpha_V": alpha_V,
        "beta_V": beta_V,
        "gamma_O2": gamma_O2,
    }

G_mission = build_stitched_digraph([graph_11, graph_12], bridge_edges)
leg0_consts = update_graph_o2_weights_for_payload(G_mission, m_samples_kg=0.0)
u_ex = next(u for u, nbrs in G_mission.adj.items() if len(nbrs) > 0)
v_ex, d_ex = next(iter(G_mission.adj[u_ex].items()))
rev_ex = G_mission.adj[v_ex][u_ex]
print(f"Unified Stitched DiGraph : {G_mission.number_of_nodes()} nodes | {G_mission.number_of_edges()} directed edges")
print(f"Asymmetric Edge Check    : {u_ex}->{v_ex} (Grade {d_ex['S_grade']:+.1f}%) = {d_ex['O2_liters']:.4f} L O2 | "
      f"Reverse ({rev_ex['S_grade']:+.1f}%) = {rev_ex['O2_liters']:.4f} L O2")

Unified Stitched DiGraph : 685 nodes | 25762 directed edges
Asymmetric Edge Check    : (230, 161)->(217, 142) (Grade +7.9%) = 2.2722 L O2 | Reverse (-7.9%) = 0.6069 L O2

---
### Step 5: Payload-Aware Directed $A^*$ & Reversed-Graph Point-of-No-Return (`G.reverse()`)
1. **Admissible 3D Metabolic Heuristic (Equation 5.1):**
   $$h_{\text{O}_2}(u, g) = \gamma_{\text{O}_2}\,d_{3D}(u, g)\Big[P_{\text{stand}}(L) + \eta_{\min}(1.05) \cdot 0.60\,\alpha_V(L)\Big] \le \Delta\text{O}_2^*(u, g)$$
2. **Reversed-Graph Single-Source Dijkstra (Equation 5.2):**
   Running Dijkstra once from the Hab node on `G_mission.rev_adj` (`G.reverse()`) computes the exact minimum return oxygen $\text{O}_{2,\text{ret}}(u)$ for every node across all stitched chunks and flags any node where $1.15\,\text{O}_{2,\text{ret}}(u) > \text{O}_{2,\text{tank}}(t)$.

In [5]:
def directed_astar_o2(G_dir: DirectedMarsGraph, start: tuple, goal: tuple, consts: dict) -> tuple:
    """Admissible 3D Metabolic Directed A* search returning (path, o2_total, dist_3d_total)."""
    min_power_floor = consts["P_stand"] + 1.05 * 0.60 * consts["alpha_V"]
    gamma_O2 = consts["gamma_O2"]
    z_goal = G_dir.nodes[goal]["z"]

    def h_o2(u: tuple) -> float:
        dz = z_goal - G_dir.nodes[u]["z"]
        d3d = math.hypot(u[0] - goal[0], u[1] - goal[1], dz)
        return gamma_O2 * d3d * min_power_floor

    g_score = {start: 0.0}
    parent = {start: None}
    pq = [(h_o2(start), 0.0, start)]

    while pq:
        f_u, g_u, u = heapq.heappop(pq)
        if u == goal:
            break
        if g_u > g_score.get(u, float("inf")):
            continue
        for v, edata in G_dir.adj[u].items():
            cand_g = g_u + edata["O2_liters"]
            if cand_g < g_score.get(v, float("inf")):
                g_score[v] = cand_g
                parent[v] = u
                heapq.heappush(pq, (cand_g + h_o2(v), cand_g, v))

    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = parent.get(curr)
    path.reverse()

    dist_3d = sum(G_dir.adj[path[i]][path[i + 1]]["d_3d"] for i in range(len(path) - 1))
    return path, g_score[goal], dist_3d

def reversed_dijkstra_return_o2(G_dir: DirectedMarsGraph, hab: tuple) -> dict:
    """Single-source Dijkstra on G.reverse() (rev_adj) computing exact return O2 to Hab from all nodes."""
    dist_o2 = {hab: 0.0}
    pq = [(0.0, hab)]
    while pq:
        d_u, u = heapq.heappop(pq)
        if d_u > dist_o2.get(u, float("inf")):
            continue
        # Traverse incoming directed edges (v -> u) via rev_adj[u][v]
        for v, edata in G_dir.rev_adj[u].items():
            cand = d_u + edata["O2_liters"]
            if cand < dist_o2.get(v, float("inf")):
                dist_o2[v] = cand
                heapq.heappush(pq, (cand, v))
    return dist_o2

pinned_all = {**graph_11["pinned_waypoints"], **graph_12["pinned_waypoints"]}
hab_node = pinned_all["SOL_3105_HAB"]
mission_legs = [
    ("SOL_3105_HAB",          "SOL_3240_WP1_CLAY",     0.0,  "Leg 1: Hab -> WP1 Clay Transition (0 kg samples)"),
    ("SOL_3240_WP1_CLAY",     "SOL_3412_WP2_PEDIMENT", 12.0, "Leg 2: WP1 -> WP2 Greenheugh Pediment (+12 kg samples)"),
    ("SOL_3412_WP2_PEDIMENT", "SOL_3950_EAST_SECTOR",  26.0, "Leg 3: WP2 -> East Chunk (1,2) Overlook (+26 kg, Cross-Chunk Migration!)"),
    ("SOL_3950_EAST_SECTOR",  "SOL_3105_HAB",          40.0, "Leg 4: East Chunk (1,2) -> Return to Hab (+40 kg heavy payload)"),
]

O2_tank_capacity_L = 22.5
O2_remaining_L = O2_tank_capacity_L
full_route_nodes = []
leg_summaries = []

for start_id, goal_id, sample_kg, label in mission_legs:
    u_start = pinned_all[start_id]
    v_goal = pinned_all[goal_id]
    consts = update_graph_o2_weights_for_payload(G_mission, m_samples_kg=sample_kg)
    path, leg_o2, leg_dist_3d = directed_astar_o2(G_mission, u_start, v_goal, consts)
    O2_remaining_L -= leg_o2

    if not full_route_nodes:
        full_route_nodes.extend(path)
    else:
        full_route_nodes.extend(path[1:])

    leg_summaries.append({
        "label": label,
        "start_id": start_id,
        "goal_id": goal_id,
        "sample_mass_kg": sample_kg,
        "total_suit_load_kg": 65.0 + sample_kg,
        "distance_3d_m": round(leg_dist_3d, 1),
        "o2_consumed_L": round(leg_o2, 2),
        "o2_tank_remaining_L": round(O2_remaining_L, 2),
        "waypoints_count": len(path),
    })
    print(f"[A* OK] {label}")
    print(f"        3D Dist: {leg_dist_3d:6.1f}m | Leg O2: {leg_o2:5.2f} L | Tank Remaining: {O2_remaining_L:6.2f} L")

# Eq 5.2: Reversed-Graph Single-Source Dijkstra for Point-of-No-Return (at peak 40kg sample load)
O2_return_map = reversed_dijkstra_return_o2(G_mission, hab_node)
pnr_nodes = [
    node for node, ret_o2 in O2_return_map.items()
    if 1.15 * ret_o2 > O2_remaining_L
]
print(f"\n[PNR SAFETY ENVELOPE] Reversed Dijkstra evaluated {len(O2_return_map)} reachable nodes in 1 pass.")
print(f"Nodes beyond Point-of-No-Return (1.15 * O2_ret > {O2_remaining_L:.2f} L remaining): {len(pnr_nodes)} nodes flagged.")

[A* OK] Leg 1: Hab -> WP1 Clay Transition (0 kg samples)
        3D Dist:   95.7m | Leg O2:  0.93 L | Tank Remaining:  21.57 L
[A* OK] Leg 2: WP1 -> WP2 Greenheugh Pediment (+12 kg samples)
        3D Dist:  154.5m | Leg O2:  5.01 L | Tank Remaining:  16.56 L
[A* OK] Leg 3: WP2 -> East Chunk (1,2) Overlook (+26 kg, Cross-Chunk Migration!)
        3D Dist:  186.2m | Leg O2:  3.32 L | Tank Remaining:  13.24 L
[A* OK] Leg 4: East Chunk (1,2) -> Return to Hab (+40 kg heavy payload)
        3D Dist:  190.8m | Leg O2:  7.78 L | Tank Remaining:   5.46 L

[PNR SAFETY ENVELOPE] Reversed Dijkstra evaluated 659 reachable nodes in 1 pass.
Nodes beyond Point-of-No-Return (1.15 * O2_ret > 5.46 L remaining): 422 nodes flagged.

---
### Step 6 & Step 7: 3D Raycast / Rover Street-View Verification & CesiumJS Mission Bundle Export
We package all 3D georeferenced outputs—including the $3\times 3$ chunk footprints, elevated 3D trajectory polyline (`[lon, lat, height_m]`), Point-of-No-Return hazard perimeter, and the **Rover Raw Images "Martian Street View" Billboards** with their Step 6.2 Sobel rock-edge density scores—into `cesium_mars_mission_bundle.json`.

In [6]:
# Build georeferenced 3D route coordinates [lon, lat, alt_m] for CesiumJS Polyline & CZML
route_coords_3d = []
for r_g, c_g in full_route_nodes:
    lon, lat = pixel_to_lonlat(np.array(r_g), np.array(c_g), meta["center_lon"], meta["center_lat"], G)
    alt_m = float(z_core_global[r_g, c_g]) + 1.2  # +1.2m torso height above DEM surface
    route_coords_3d.append([round(float(lon), 6), round(float(lat), 6), round(alt_m, 2)])

# Build 3x3 chunk bounding boxes in (west, south, east, north) degrees for CesiumJS Rectangle entities
chunk_footprints = []
for I in range(3):
    for J in range(3):
        lon_w, lat_n = pixel_to_lonlat(np.array(I * L), np.array(J * L), meta["center_lon"], meta["center_lat"], G)
        lon_e, lat_s = pixel_to_lonlat(np.array((I + 1) * L), np.array((J + 1) * L), meta["center_lon"], meta["center_lat"], G)
        chunk_footprints.append({
            "chunk_ij": [I, J],
            "is_center_start": (I == 1 and J == 1),
            "is_activated": [I, J] in [[1, 1], [1, 2]],
            "west_deg": round(float(lon_w), 6),
            "south_deg": round(float(lat_s), 6),
            "east_deg": round(float(lon_e), 6),
            "north_deg": round(float(lat_n), 6),
        })

# Export Point-of-No-Return (PNR) flagged vertices for CesiumJS red hazard point cloud / polygon
pnr_geojson_points = []
for r_g, c_g in pnr_nodes[:250]:
    lon, lat = pixel_to_lonlat(np.array(r_g), np.array(c_g), meta["center_lon"], meta["center_lat"], G)
    pnr_geojson_points.append({
        "lon": round(float(lon), 6),
        "lat": round(float(lat), 6),
        "alt_m": round(float(z_core_global[r_g, c_g]), 2),
        "return_o2_required_L": round(float(O2_return_map[(r_g, c_g)]), 2),
    })

cesium_bundle = {
    "cesium_version": "1.114+",
    "mission_metadata": meta,
    "chunk_grid_3x3": chunk_footprints,
    "eva_mission_legs": leg_summaries,
    "eva_3d_route_lon_lat_alt": route_coords_3d,
    "point_of_no_return_envelope": {
        "o2_tank_capacity_L": O2_tank_capacity_L,
        "o2_end_of_mission_L": round(O2_remaining_L, 2),
        "safety_reserve_factor": 1.15,
        "flagged_node_count": len(pnr_nodes),
        "hazard_nodes": pnr_geojson_points,
    },
    "martian_streetview_rover_images": rover_catalog,
}

with open("cesium_mars_mission_bundle.json", "w", encoding="utf-8") as f:
    json.dump(cesium_bundle, f, indent=2)

print("[CESIUM EXPORT COMPLETE] -> cesium_mars_mission_bundle.json")
print(f"  - 3x3 Chunk Footprints      : {len(chunk_footprints)} chunks (Center (1,1) + Migrated East (1,2) active)")
print(f"  - 3D Trajectory Waypoints   : {len(route_coords_3d)} [lon, lat, alt] vertices")
print(f"  - Rover Street-View Images  : {len(rover_catalog)} georeferenced NASA Navcam/Mastcam nodes")

[CESIUM EXPORT COMPLETE] -> cesium_mars_mission_bundle.json
  - 3x3 Chunk Footprints      : 9 chunks (Center (1,1) + Migrated East (1,2) active)
  - 3D Trajectory Waypoints   : 36 [lon, lat, alt] vertices
  - Rover Street-View Images  : 6 georeferenced NASA Navcam/Mastcam nodes